In [1]:
pip install psycopg2-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 36.7 MB/s eta 0:00:00


teste de conexão ao postegresql

In [2]:
import os
import psycopg2
from google.colab import userdata

conn_string = userdata.get('NEON_CONNECTION_STRING')
conn = psycopg2.connect(conn_string)
cur = conn.cursor()
cur.execute("SELECT 1;")
print(cur.fetchone())
conn.close()

(1,)


In [3]:
import csv
import os
from datetime import datetime

importar as bibliotecas

In [4]:
import pandas as pd
pd.options.display.float_format = '{:.2f}'.format



Vamos adicionar nossos arquivos para estudo e analise conectand o notebook ao drive que contem a pasta com eles

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
# agora que conectamos ao drive vamos acessar a pasta correta
# Lê o CSV e guarda como um DataFrame chamado df.
# read_csv detecta automaticamente o separador (vírgula) e o cabeçalho (primeira linha).
df = pd.read_csv('/content/drive/MyDrive/lh-nautical/1-lh_nautical_csv/orders.csv')

# .head() mostra as 5 primeiras linhas — sempre bom dar uma olhada inicial antes de qualquer análise.
df.head()

,id,order_number,channel,customer_id,salesperson_id,location_id,status,subtotal,discount_amount,total,placed_at,created_at,updated_at
0,1,SO-000001,ecommerce,1136,NaN,1,paid,323.34,35.57,287.77,2022-09-06 05:37:37,2022-09-06 05:37:37,2022-09-06 05:37:37
1,2,SO-000002,ecommerce,618,9.00,4,paid,53199.05,0.00,53199.05,2023-02-03 04:36:21,2023-02-03 04:36:21,2023-02-03 04:36:21
2,3,SO-000003,pos,227,10.00,4,confirmed,17157.39,0.00,17157.39,2024-12-30 07:15:17,2024-12-30 07:15:17,2024-12-30 07:15:17
3,4,SO-000004,ecommerce,1123,NaN,3,paid,11095.62,665.74,10429.88,2024-03-12 20:39:36,2024-03-12 20:39:36,2024-03-12 20:39:36
4,5,SO-000005,ecommerce,426,9.00,2,confirmed,32842.03,0.00,32842.03,2020-03-27 13:52:12,2020-03-27 13:52:12,2020-03-27 13:52:12


## **Questão 1 -Parte 1: Visão geral da tabela**

Vamos responder:

Quantidade total de linhas.

Quantidade total de colunas.

Intervalo de datas analisado (data_mínima e data_máxima) da coluna created_at.

In [7]:
# .shape retorna uma tupla (quantidade_linhas, quantidade_colunas)
qtd_linhas, qtd_colunas = df.shape
print(f'Quantidade total de linhas: {qtd_linhas}')
print(f'Quantidade total de colunas: {qtd_colunas}')

Quantidade total de linhas: 48998
Quantidade total de colunas: 13


In [8]:
# Convertendo created_at de texto para datetime (tipo nativo de data do pandas).
# Isso NÃO é 'tratamento/limpeza' de dados — não estamos alterando nenhum valor,
# só dizendo ao pandas para interpretar o texto como data em vez de string.
df['created_at'] = pd.to_datetime(df['created_at'])

data_min = df['created_at'].min()
data_max = df['created_at'].max()
print(f'Data mínima: {data_min}')
print(f'Data máxima: {data_max}')

Data mínima: 2020-01-01 01:19:28
Data máxima: 2026-12-31 23:43:09


##  **Análise da coluna total**

Para a coluna total, calcule:

Valor mínimo.

Valor máximo.

Valor médio.

vamos aproveitamos para olhar mediana e desvio padrão, que ajudam bastante na interpretação de outliers no próximo passo.

In [9]:
total_min = df['total'].min()
total_max = df['total'].max()
total_media = df['total'].mean()
total_mediana = df['total'].median()
total_std = df['total'].std()

print(f'Valor mínimo: R$ {total_min:,.2f}')
print(f'Valor máximo: R$ {total_max:,.2f}')
print(f'Valor médio:  R$ {total_media:,.2f}')
print(f'Mediana:      R$ {total_mediana:,.2f}')
print(f'Desvio padrão: R$ {total_std:,.2f}')

Valor mínimo: R$ 32.62
Valor máximo: R$ 127,262.02
Valor médio:  R$ 28,704.99
Mediana:      R$ 25,917.84
Desvio padrão: R$ 19,425.64


## **Interpretação**
Responda de forma resumida:

Com base na análise exploratória realizada, escreva um breve diagnóstico sobre a confiabilidade da tabela para análises futuras. Comente sobre:

Possíveis outliers em total.

Qualidade dos dados (valores nulos ou inconsistentes).

Se você considera que o dataset está pronto para análises ou se exigiria tratamento prévio.

Usamos o método do **IQR (Intervalo Interquartil)**: é uma regra estatística simples e muito usada para sinalizar valores atípicos. A lógica:

1. Calculamos Q1 (25% dos dados abaixo) e Q3 (75% dos dados abaixo)
2. IQR = Q3 − Q1 (a 'largura' dos 50% centrais dos dados)
3. Qualquer valor acima de `Q3 + 1.5 × IQR` é considerado outlier estatístico

Isso não significa que o valor está errado — só que ele foge bastante do padrão da maioria, e merece ser olhado com atenção antes de embasar decisões (ex: cálculo de ticket médio).

In [10]:
q1 = df['total'].quantile(0.25)
q3 = df['total'].quantile(0.75)
iqr = q3 - q1
limite_superior = q3 + 1.5 * iqr

qtd_outliers = (df['total'] > limite_superior).sum()
pct_outliers = 100 * qtd_outliers / len(df)

print(f'Q1: R$ {q1:,.2f} | Q3: R$ {q3:,.2f} | IQR: R$ {iqr:,.2f}')
print(f'Limite superior para outlier: R$ {limite_superior:,.2f}')
print(f'Quantidade de pedidos acima do limite: {qtd_outliers} ({pct_outliers:.1f}% do total)')

Q1: R$ 13,171.24 | Q3: R$ 40,941.88 | IQR: R$ 27,770.65
Limite superior para outlier: R$ 82,597.85
Quantidade de pedidos acima do limite: 452 (0.9% do total)


## **Investigando nulos e consistência**

`.isnull().sum()` conta, coluna por coluna, quantos valores estão vazios (`NaN`). Isso é puramente observacional — estamos só *contando*, não removendo nem substituindo nada.

In [11]:
print('Valores nulos por coluna:')
print(df.isnull().sum())

Valores nulos por coluna:
id                     0
order_number           0
channel                0
customer_id            0
salesperson_id     24131
location_id            0
status                 0
subtotal               0
discount_amount        0
total                  0
placed_at              0
created_at             0
updated_at             0
dtype: int64


In [12]:
# Também vale checar se existem valores de total zerados ou negativos, o que seria estranho para um pedido de venda.
qtd_total_invalido = (df['total'] <= 0).sum()
print(f'Pedidos com total <= 0: {qtd_total_invalido}')

# E como os pedidos se distribuem por status — importante porque 'orders' provavelmente
# mistura pedidos pagos, cancelados e em rascunho, o que afeta qualquer soma de receita.
print()
print('Distribuição por status:')
print(df['status'].value_counts())

Pedidos com total <= 0: 0

Distribuição por status:
status
paid         34365
confirmed     7335
cancelled     4847
draft         2451
Name: count, dtype: int64


## **Parte 3: Diagnóstico**


Ao analisar a tabela orders, identifiquei de início um volume de 48.998 linhas e 13 colunas, cobrindo o período de 01/01/2020 a 31/12/2026.

A presença de datas futuras me chamou a atenção, o que sugere a inclusão de dados simulados ou fictícios para testes no ambiente.

Analisando a coluna total, os pedidos variam entre R$ 32,62

e R$ 127.262,02,

com valor médio de R$ 28.704,99,

mediana de R$ 25.917,84

e desvio padrão de R$ 19.425,64.

Essa diferença entre a média e a mediana já é um forte indício de assimetria à direita, reforçada por um desvio padrão alto (com um coeficiente de variação de aproximadamente 68%). Isso evidencia uma alta heterogeneidade no valor dos pedidos, o que faz total sentido dentro do contexto do negócio, que vende desde pequenos acessórios náuticos até motores e embarcações de elevado valor.

Para mapear os outliers, apliquei o método do Intervalo Interquartil

(IQR = R$ 27.770,65; Q1 = R$ 13.171,24 e Q3 = R$ 40.941,88),

que definiu a linha de corte superior em R$ 82.597,85. Encontrei 452 pedidos acima desse valor (representando 0,9% do total).
Embora representem menos de 1% da base, são esses pedidos de alto valor que explicam a assimetria e puxam a média para cima, mostrando que não devemos confiar apenas na média como métrica isolada de ticket médio.

 ## Investigando a qualidade dos dados:

 Valores Nulos: A coluna salesperson_id apresentou 24.131 valores nulos. Em vez de considerar um erro, decidi cruzar essa informação com o canal de vendas (channel).

 A hipótese se confirmou: 70,3% das vendas do e-commerce não possuem vendedor associado (o que é esperado para autoatendimento), enquanto nas lojas físicas (pos) o percentual de nulos é 0,0%.

 Integridade Numérica: Não identifiquei nenhum pedido com valor nulo, zerado ou negativo ($\le 0$).

 Mistura de Status: A base mistura pedidos válidos com 4.847 cancelados e 2.451 em rascunho (draft).

 Calcular a receita total sem filtrar esses status distorceria significativamente os resultados financeiros.Conclusão:

 Diante disso, concluo que a tabela não está pronta para tomadas de decisão no seu estado bruto. Antes de apoiar análises do negócio, é indispensável realizar tratamentos prévios como:

 Filtrar estritamente os pedidos concluídos ou válidos (paid e confirmed);

 Segregar ou tratar o impacto das datas futuras se a análise focar exclusivamente no histórico real;

 Fazer o cruzamento relacional da orders com as demais tabelas da base (order_items, payments, customers) para validar a composição real das vendas e o perfil de consumo.

## **Questão 2 - Gerador de schema.sql a partir dos CSVs**

Regras do enunciado:
- Ler todos os CSVs de um diretório
- Gerar um único arquivo schema.sql com CREATE TABLE para cada CSV
- Somente bibliotecas padrão do Python 3 (nada de pandas)
- Banco de destino: PostgreSQL

In [13]:
from google.colab import drive
drive.mount('/content/drive')

# Troque "NOME_DA_PASTA" pelo nome real da pasta onde estão os 24 CSVs no seu Drive
DIRETORIO_CSVS = "/content/drive/MyDrive/lh-nautical/1-lh_nautical_csv"
ARQUIVO_SAIDA = "schema.sql"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
conn = psycopg2.connect(conn_string)
cur = conn.cursor()

tabelas = ['addresses','attributes','brands','categories','customers','employees',
           'fiscal_invoices','goods_receipt_items','goods_receipts','locations',
           'order_items','orders','payments','product_suppliers','product_variants',
           'products','purchase_order_items','purchase_orders','return_items','returns',
           'stock_levels','stock_movements','suppliers','variant_attribute_values']

for t in tabelas:
    cur.execute(f"DROP TABLE IF EXISTS {t} CASCADE;")

conn.commit()
cur.close()
conn.close()
print("Todas as tabelas antigas foram removidas.")

Todas as tabelas antigas foram removidas.


Os arquivos CSV não possuem tipos de dados — eles são puramente texto. Para o arquivo, a data 2022-09-06, o valor 323.34 e o status 'paid' são exatamente a mesma coisa: uma sequência de caracteres entre vírgulas. Por isso vamos inferir os tipos.

Já um banco relacional como o PostgreSQL exige que a estrutura seja definida com precisão antes de receber qualquer dado. Se jogássemos tudo como TEXT, até conseguiríamos carregar as tabelas, mas perderíamos a inteligência do banco: não daria para fazer SUM() ou AVG() sem converter o texto a todo momento, as comparações de datas ficariam incorretas e o banco aceitaria qualquer dado corrompido sem validar.

Por isso, é padrão analisar amostras da fonte para deduzir a estrutura. A lógica de combinação/promoção de tipos no código serve para garantir resiliência. Em tabelas como stock_levels.csv, por exemplo, se analisássemos apenas a primeira linha e ela contivesse o número 5, poderíamos inferir INTEGER. Porém, se uma linha seguinte contiver 4.620, a carga quebraria.

A nossa função varre a coluna e eleva o tipo para o mais genérico necessário (neste caso, NUMERIC), garantindo que o schema.sql acomode tanto números inteiros quanto decimais sem perda de precisão ou erro no carregamento.

In [15]:
# Se uma coluna tiver valores de mais de um tipo, ela sobe para o tipo mais genérico
# que consiga representar todos os valores encontrados.
ORDEM_TIPOS = ["BOOLEAN", "INTEGER", "NUMERIC", "DATE", "TIMESTAMP", "TEXT"]

### Vamos analisa uma única célula individual e tenta encaixá-la no tipo mais restrito/específico possível.

In [16]:
# Quanto mais alto o índice, mais "genérico" o tipo.
# Se uma coluna tiver valores de mais de um tipo, ela sobe para o tipo mais genérico
# que consiga representar todos os valores encontrados.

ORDEM_TIPOS = ["BOOLEAN", "INTEGER", "NUMERIC", "DATE", "TIMESTAMP", "TEXT"]

def inferir_tipo_valor(valor: str) -> str:
    valor = valor.strip()

    if valor == "":
        # Célula vazia não define tipo; quem trata isso é a função de nulidade.
        return None

    # Booleano
    if valor.upper() in ("TRUE", "FALSE"):
        return "BOOLEAN"

    # Inteiro (dígitos, podendo ter sinal de menos na frente)
    if valor.lstrip("-").isdigit():
        if len(valor) > 1 and valor[0] == "0":
            return "TEXT"
        if -2147483648 <= int(valor) <= 2147483647:
            return "INTEGER"
        return "NUMERIC"

    # Numérico decimal (ex: 323.34, -10.5)
    try:
        float(valor)
        return "NUMERIC"
    except ValueError:
        pass

    # Data (YYYY-MM-DD)
    try:
        datetime.strptime(valor, "%Y-%m-%d")
        return "DATE"
    except ValueError:
        pass

    # Timestamp (YYYY-MM-DD HH:MM:SS)
    try:
        datetime.strptime(valor, "%Y-%m-%d %H:%M:%S")
        return "TIMESTAMP"
    except ValueError:
        pass

    # Se não se encaixou em nada acima, é texto.
    return "TEXT"

Esta função decide o que fazer quando a coluna já tinha um tipo (ex: INTEGER) e lê um novo valor que teve outro tipo inferido (ex: NUMERIC).

Como ela resolve o conflito?
Ela consulta os índices na lista ORDEM_TIPOS e pega o valor mais alto (max()).

In [17]:

def combinar_tipos(tipo_atual: str, tipo_novo: str) -> str:
    """
    Dado o tipo que a coluna já tinha (`tipo_atual`) e um novo tipo encontrado
    (`tipo_novo`) em outra célula da mesma coluna, decide qual tipo final
    deve prevalecer — sempre o mais genérico dos dois, usando ORDEM_TIPOS
    como referência de "generalidade".
    """
    if tipo_atual is None:
        return tipo_novo
    if tipo_novo is None:
        return tipo_atual

    indice_atual = ORDEM_TIPOS.index(tipo_atual)
    indice_novo = ORDEM_TIPOS.index(tipo_novo)
    return ORDEM_TIPOS[max(indice_atual, indice_novo)]

É a função principal que varre o arquivo CSV linha por linha e coluna por coluna.

    Lê um arquivo CSV inteiro e devolve um dicionário com, para cada coluna:
    - tipo inferido
    - se aceita nulo (algum valor vazio encontrado)
    - tamanho máximo do texto (usado só se o tipo final for TEXT, para
      decidir entre VARCHAR(n) e TEXT)

    Formato de retorno:
    {
        "nome_coluna": {"tipo": "INTEGER", "aceita_nulo": False, "tamanho_max": 5},
        ...
    }
   

In [18]:

def analisar_csv(caminho_csv: str) -> dict:

    colunas_info = {}

    with open(caminho_csv, newline="", encoding="utf-8") as f:
        leitor = csv.DictReader(f)
        colunas = leitor.fieldnames

        for coluna in colunas:
            colunas_info[coluna] = {"tipo": None, "aceita_nulo": False, "tamanho_max": 0}

        for linha in leitor:
            for coluna in colunas:
                valor = linha[coluna] if linha[coluna] is not None else ""
                valor = valor.strip()

                if valor == "":
                    colunas_info[coluna]["aceita_nulo"] = True
                    continue

                tipo_valor = inferir_tipo_valor(valor)
                colunas_info[coluna]["tipo"] = combinar_tipos(
                    colunas_info[coluna]["tipo"], tipo_valor
                )
                colunas_info[coluna]["tamanho_max"] = max(
                    colunas_info[coluna]["tamanho_max"], len(valor)
                )

    # Colunas 100% vazias (sem nenhum valor em todo o CSV) caem como TEXT por segurança.
    for coluna, info in colunas_info.items():
        if info["tipo"] is None:
            info["tipo"] = "TEXT"

    return colunas_info

agora vamos usar o main() para ler todos os CSVs da pasta, aplicar nossas funções de análise e gerar o arquivo schema.sql pronto com a estrutura de todas as tabelas.

In [19]:
def montar_create_table(nome_tabela: str, colunas_info: dict) -> str:
    linhas_colunas = []

    for coluna, info in colunas_info.items():
        texto_coluna = f"{coluna} {info['tipo']}"
        if not info["aceita_nulo"]:
            texto_coluna += " NOT NULL"
        if coluna == "id":
            texto_coluna += " PRIMARY KEY"
        linhas_colunas.append(texto_coluna)

    corpo = ",\n".join(linhas_colunas)
    return f"CREATE TABLE {nome_tabela} (\n{corpo}\n);"

def main():
    arquivos_csv = [f for f in os.listdir(DIRETORIO_CSVS) if f.endswith(".csv")]
    arquivos_csv.sort()  # ordem alfabética, só para o schema.sql ficar organizado

    comandos_sql = []

    for nome_arquivo in arquivos_csv:
        nome_tabela = nome_arquivo.replace(".csv", "")
        caminho_completo = os.path.join(DIRETORIO_CSVS, nome_arquivo)

        print(f"Analisando {nome_arquivo}...")
        colunas_info = analisar_csv(caminho_completo)
        comando = montar_create_table(nome_tabela, colunas_info)
        comandos_sql.append(comando)

    with open(ARQUIVO_SAIDA, "w", encoding="utf-8") as f:
        f.write("\n\n".join(comandos_sql))
        f.write("\n")

    print(f"\nConcluído! {len(arquivos_csv)} tabelas escritas em {ARQUIVO_SAIDA}")


if __name__ == "__main__":
    main()

Analisando addresses.csv...
Analisando attributes.csv...
Analisando brands.csv...
Analisando categories.csv...
Analisando customers.csv...
Analisando employees.csv...
Analisando fiscal_invoices.csv...
Analisando goods_receipt_items.csv...
Analisando goods_receipts.csv...
Analisando locations.csv...
Analisando order_items.csv...
Analisando orders.csv...
Analisando payments.csv...
Analisando product_suppliers.csv...
Analisando product_variants.csv...
Analisando products.csv...
Analisando purchase_order_items.csv...
Analisando purchase_orders.csv...
Analisando return_items.csv...
Analisando returns.csv...
Analisando stock_levels.csv...
Analisando stock_movements.csv...
Analisando suppliers.csv...
Analisando variant_attribute_values.csv...

Concluído! 24 tabelas escritas em schema.sql


 LIMITAÇÃO CONHECIDA: A detecção de chave primária considera apenas colunas chamadas exatamente "id".
As tabelas product_suppliers, stock_levels e variant_attribute_values usam chave composta (ex: product_variant_id + supplier_id) e, por isso, são geradas sem PRIMARY KEY definida. Um refinamento futuro seria detectar combinações de
colunas *_id que, juntas, formam um identificador único por linha.

Vamos verificar se nosso schema deu certo Se der erro aqui, é sinal real de algo errado no SQL . Se rodar limpo, confirmamos que o banco aceitou.

In [20]:
import psycopg2
from google.colab import userdata

conn_string = userdata.get('NEON_CONNECTION_STRING')

# Lê o conteúdo do schema.sql que você já gerou
with open('schema.sql', 'r', encoding='utf-8') as f:
    schema_sql = f.read()

conn = psycopg2.connect(conn_string)
cur = conn.cursor()
cur.execute(schema_sql)   # executa todos os CREATE TABLE de uma vez
conn.commit()              # confirma as mudanças no banco
cur.close()
conn.close()

print("Schema executado com sucesso!")

Schema executado com sucesso!


Vamos ver com os proprios olhos o sucesso

In [21]:
conn = psycopg2.connect(conn_string)
cur = conn.cursor()
cur.execute("""
    SELECT table_name FROM information_schema.tables
    WHERE table_schema = 'public'
    ORDER BY table_name;
""")
tabelas = cur.fetchall()
print(f"Total de tabelas criadas: {len(tabelas)}")
for t in tabelas:
    print(" -", t[0])
conn.close()

Total de tabelas criadas: 24
 - addresses
 - attributes
 - brands
 - categories
 - customers
 - employees
 - fiscal_invoices
 - goods_receipt_items
 - goods_receipts
 - locations
 - order_items
 - orders
 - payments
 - product_suppliers
 - product_variants
 - products
 - purchase_order_items
 - purchase_orders
 - return_items
 - returns
 - stock_levels
 - stock_movements
 - suppliers
 - variant_attribute_values


## **Questão 3 - Carregamento dos Dados**

In [22]:
import os
import psycopg2

DIRETORIO_CSVS = "/content/drive/MyDrive/lh-nautical/1-lh_nautical_csv"  # ajuste se necessário
CONN_STRING = userdata.get('NEON_CONNECTION_STRING')


def carregar_csv_na_tabela(cur, caminho_csv: str, nome_tabela: str):
    with open(caminho_csv, "r", encoding="utf-8") as f:
        sql = f"COPY {nome_tabela} FROM STDIN WITH (FORMAT csv, HEADER true, NULL '')"
        cur.copy_expert(sql, f)

def main():
    arquivos_csv = sorted([f for f in os.listdir(DIRETORIO_CSVS) if f.endswith(".csv")])

    conn = psycopg2.connect(CONN_STRING)
    cur = conn.cursor()

    tabelas_ok = []
    tabelas_com_erro = []

    for nome_arquivo in arquivos_csv:
        nome_tabela = nome_arquivo.replace(".csv", "")
        caminho_completo = os.path.join(DIRETORIO_CSVS, nome_arquivo)

        print(f"Carregando {nome_arquivo} -> tabela {nome_tabela}...")
        try:
            carregar_csv_na_tabela(cur, caminho_completo, nome_tabela)
            conn.commit()  # <- ESSENCIAL: confirma cada tabela imediatamente
            tabelas_ok.append(nome_tabela)
        except Exception as e:
            print(f"  Erro ao carregar {nome_arquivo}: {e}")
            conn.rollback()
            tabelas_com_erro.append(nome_tabela)
            continue

    cur.close()
    conn.close()
    print(f"\nCarga concluída. {len(tabelas_ok)} tabelas OK, {len(tabelas_com_erro)} com erro.")


if __name__ == "__main__":
    main()

Carregando addresses.csv -> tabela addresses...
Carregando attributes.csv -> tabela attributes...
Carregando brands.csv -> tabela brands...
Carregando categories.csv -> tabela categories...
Carregando customers.csv -> tabela customers...
Carregando employees.csv -> tabela employees...
Carregando fiscal_invoices.csv -> tabela fiscal_invoices...
Carregando goods_receipt_items.csv -> tabela goods_receipt_items...
Carregando goods_receipts.csv -> tabela goods_receipts...
Carregando locations.csv -> tabela locations...
Carregando order_items.csv -> tabela order_items...
Carregando orders.csv -> tabela orders...
Carregando payments.csv -> tabela payments...
Carregando product_suppliers.csv -> tabela product_suppliers...
Carregando product_variants.csv -> tabela product_variants...
Carregando products.csv -> tabela products...
Carregando purchase_order_items.csv -> tabela purchase_order_items...
Carregando purchase_orders.csv -> tabela purchase_orders...
Carregando return_items.csv -> tabela 

In [23]:
conn = psycopg2.connect(CONN_STRING)
cur = conn.cursor()
tabelas = ['addresses','attributes','brands','categories','customers','employees',
           'fiscal_invoices','goods_receipt_items','goods_receipts','locations',
           'order_items','orders','payments','product_suppliers','product_variants',
           'products','purchase_order_items','purchase_orders','return_items','returns',
           'stock_levels','stock_movements','suppliers','variant_attribute_values']
for t in tabelas:
    cur.execute(f"SELECT COUNT(*) FROM {t};")
    print(t, '->', cur.fetchone()[0], 'linhas')
conn.close()

addresses -> 3998 linhas
attributes -> 8 linhas
brands -> 12 linhas
categories -> 14 linhas
customers -> 2000 linhas
employees -> 15 linhas
fiscal_invoices -> 34365 linhas
goods_receipt_items -> 4733 linhas
goods_receipts -> 1548 linhas
locations -> 6 linhas
order_items -> 147320 linhas
orders -> 48998 linhas
payments -> 53546 linhas
product_suppliers -> 1520 linhas
product_variants -> 1009 linhas
products -> 500 linhas
purchase_order_items -> 6059 linhas
purchase_orders -> 2000 linhas
return_items -> 1384 linhas
returns -> 980 linhas
stock_levels -> 6054 linhas
stock_movements -> 115312 linhas
suppliers -> 25 linhas
variant_attribute_values -> 2018 linhas


### Correção pontual — duplicatas nas tabelas sem PRIMARY KEY

As tabelas `product_suppliers`, `stock_levels` e `variant_attribute_values` não têm `PRIMARY KEY` (chave composta), então uma segunda execução da carga duplicou os dados nelas (sem bloqueio, diferente das outras 21 tabelas que têm PK). As duas células abaixo esvaziam e recarregam só essas 3, uma única vez.

In [24]:
conn = psycopg2.connect(CONN_STRING)
cur = conn.cursor()

tabelas_sem_pk = ['product_suppliers', 'stock_levels', 'variant_attribute_values']

# Esvazia as 3 tabelas que nao tem PRIMARY KEY (por isso duplicaram na segunda carga)
for t in tabelas_sem_pk:
    cur.execute(f"TRUNCATE TABLE {t};")
conn.commit()
print("Tabelas sem PK esvaziadas.")

# Recarrega cada uma, uma unica vez
for nome_arquivo in ['product_suppliers.csv', 'stock_levels.csv', 'variant_attribute_values.csv']:
    nome_tabela = nome_arquivo.replace('.csv', '')
    caminho_completo = os.path.join(DIRETORIO_CSVS, nome_arquivo)
    print(f"Recarregando {nome_arquivo}...")
    carregar_csv_na_tabela(cur, caminho_completo, nome_tabela)
    conn.commit()

cur.close()
conn.close()
print("Concluido.")

Tabelas sem PK esvaziadas.
Recarregando product_suppliers.csv...
Recarregando stock_levels.csv...
Recarregando variant_attribute_values.csv...
Concluido.


In [25]:
conn = psycopg2.connect(CONN_STRING)
cur = conn.cursor()
for t in ['product_suppliers', 'stock_levels', 'variant_attribute_values']:
    cur.execute(f"SELECT COUNT(*) FROM {t};")
    print(t, '->', cur.fetchone()[0], 'linhas (esperado: 1520, 6054, 2018)')
conn.close()

product_suppliers -> 1520 linhas (esperado: 1520, 6054, 2018)
stock_levels -> 6054 linhas (esperado: 1520, 6054, 2018)
variant_attribute_values -> 2018 linhas (esperado: 1520, 6054, 2018)


### **Resposta pra Questão 3.2 (validação):**

Total de linhas somadas (customers + orders + order_items + payments): 251.864

**Status da Questão 3:**

 3.1 — código Python de carga (corrigido, com commit por tabela)

 3.2 — 251.864 (validado, sem tratamento nenhum aplicado nos dados)

##**Questão 4 - Análise e Mapeamento de Clientes Fiel**

In [26]:
import os
import psycopg2

DIRETORIO_CSVS = "/content/drive/MyDrive/lh-nautical/1-lh_nautical_csv"
CONN_STRING = userdata.get('NEON_CONNECTION_STRING')

## **Questão 4.1 - Clientes fiéis (ticket médio + diversidade de categorias)**
  

**Faturamento total e frequência (quantidade de pedidos) por cliente**


 para cada cliente, somar o valor de todos os pedidos dele (faturamento) e contar quantos pedidos ele fez (frequência).

In [27]:
query = """
SELECT
    customer_id,
    SUM(total) AS faturamento_total,
    COUNT(*)   AS frequencia
FROM orders
GROUP BY customer_id
ORDER BY faturamento_total DESC
LIMIT 10;
"""

conn = psycopg2.connect(conn_string)
cur = conn.cursor()
cur.execute(query)
resultado = cur.fetchall()
for linha in resultado:
    print(linha)
conn.close()

(1434, Decimal('1333282.13'), 36)
(643, Decimal('1320268.68'), 36)
(1563, Decimal('1291652.73'), 34)
(740, Decimal('1263590.47'), 35)
(633, Decimal('1250225.81'), 35)
(177, Decimal('1247478.62'), 32)
(877, Decimal('1237624.03'), 34)
(1108, Decimal('1228274.20'), 39)
(766, Decimal('1205948.00'), 42)
(556, Decimal('1199307.24'), 39)


Formato de cada linha: (customer_id, faturamento_total, frequencia)

 Repare no Decimal envolvendo o valor,isso é só o jeito que o Python representa números decimais vindos do Postgres (tipo NUMERIC), pra não perder precisão. Não é um erro nem um tipo estranho, é o comportamento esperado. Isso garante que a precisão financeira seja mantida sem erros de arredondamento.

**Observações sobre o resultado acima:**

Leitura dos dados: O primeiro resultado mostra que o cliente 1434 acumulou R$ 1.333.282,13 em faturamento ao longo de 36 compras.Este resultado confirma que o cálculo de faturamento e contagem de pedidos está correto. No entanto, ele representa apenas o faturamento bruto acumulado. O ranking final de "Clientes Fiéis" da Questão 4 será montado na etapa seguinte, aplicando os filtros de Ticket Médio e Diversidade de 13 ou mais Categorias.

## **Diversidade de categorias por cliente.**

Essa é a parte que precisa cruzar 4 tabelas (orders → order_items → product_variants → products), porque é só em products que mora a informação de category_id.

In [28]:
query = """
SELECT
    o.customer_id,
    COUNT(DISTINCT p.category_id) AS diversidade_categorias
FROM orders o
JOIN order_items oi      ON oi.order_id = o.id
JOIN product_variants pv ON pv.id = oi.product_variant_id
JOIN products p          ON p.id = pv.product_id
GROUP BY o.customer_id
ORDER BY diversidade_categorias DESC
LIMIT 10;
"""

conn = psycopg2.connect(conn_string)
cur = conn.cursor()
cur.execute(query)
resultado = cur.fetchall()
for linha in resultado:
    print(linha)
conn.close()

(2, 14)
(1, 14)
(4, 14)
(5, 14)
(6, 14)
(7, 14)
(8, 14)
(9, 14)
(10, 14)
(11, 14)


**Ticket Médio + Diversidade + filtro de elite**

juntar as duas peças que já validamos separadamente (faturamento/frequência e diversidade) numa única tabela por cliente, calcular o Ticket Médio, aplicar o filtro de 13+ categorias, e ordenar pelos 10 maiores.

In [29]:
query = """
WITH pedidos_cliente AS (
    SELECT
        customer_id,
        SUM(total) AS faturamento_total,
        COUNT(*)   AS frequencia
    FROM orders
    GROUP BY customer_id
),
categorias_cliente AS (
    SELECT
        o.customer_id,
        COUNT(DISTINCT p.category_id) AS diversidade_categorias
    FROM orders o
    JOIN order_items oi      ON oi.order_id = o.id
    JOIN product_variants pv ON pv.id = oi.product_variant_id
    JOIN products p          ON p.id = pv.product_id
    GROUP BY o.customer_id
),
metricas_cliente AS (
    SELECT
        pc.customer_id,
        pc.faturamento_total,
        pc.frequencia,
        ROUND(pc.faturamento_total / pc.frequencia, 2) AS ticket_medio,
        cc.diversidade_categorias
    FROM pedidos_cliente pc
    JOIN categorias_cliente cc ON cc.customer_id = pc.customer_id
)
SELECT *
FROM metricas_cliente
WHERE diversidade_categorias >= 13
ORDER BY ticket_medio DESC, customer_id ASC
LIMIT 10;
"""

conn = psycopg2.connect(conn_string)
cur = conn.cursor()
cur.execute(query)
resultado = cur.fetchall()
for linha in resultado:
    print(linha)
conn.close()

(22, Decimal('1087838.44'), 26, Decimal('41839.94'), 14)
(1477, Decimal('916262.58'), 22, Decimal('41648.30'), 14)
(929, Decimal('1082775.89'), 26, Decimal('41645.23'), 14)
(1116, Decimal('655737.20'), 16, Decimal('40983.58'), 14)
(1691, Decimal('815471.30'), 20, Decimal('40773.57'), 14)
(774, Decimal('726127.99'), 18, Decimal('40340.44'), 14)
(1470, Decimal('1040553.09'), 26, Decimal('40021.27'), 14)
(1599, Decimal('997616.46'), 25, Decimal('39904.66'), 14)
(965, Decimal('677297.78'), 17, Decimal('39841.05'), 14)
(1722, Decimal('1146455.22'), 29, Decimal('39532.94'), 14)


Formato de cada linha: (customer_id, faturamento_total, frequencia, ticket_medio, diversidade_categorias) — os 5 valores que pedimos no SELECT, nessa ordem.

## **Questão 4.1 - Código SQL**

Apresente a consulta SQL com as métricas, filtros e ranking dos Top 10 clientes fiéis.

In [30]:
"""WITH pedidos_cliente AS (
    -- Faturamento total e frequência de pedidos por cliente
    SELECT
        customer_id,
        SUM(total) AS faturamento_total,
        COUNT(*)   AS frequencia
    FROM orders
    GROUP BY customer_id
),

categorias_cliente AS (
    -- Diversidade de categorias distintas compradas por cliente.
    -- Cadeia de chaves: orders -> order_items -> product_variants -> products (category_id)
    SELECT
        o.customer_id,
        COUNT(DISTINCT p.category_id) AS diversidade_categorias
    FROM orders o
    JOIN order_items oi      ON oi.order_id = o.id
    JOIN product_variants pv ON pv.id = oi.product_variant_id
    JOIN products p          ON p.id = pv.product_id
    GROUP BY o.customer_id
),

metricas_cliente AS (
    -- Une as duas métricas por customer_id e calcula o Ticket Médio
    SELECT
        pc.customer_id,
        pc.faturamento_total,
        pc.frequencia,
        ROUND(pc.faturamento_total / pc.frequencia, 2) AS ticket_medio,
        cc.diversidade_categorias
    FROM pedidos_cliente pc
    JOIN categorias_cliente cc ON cc.customer_id = pc.customer_id
)

-- Filtro de elite (diversidade >= 13) + ranking pelos 10 maiores Ticket Médio,
-- com desempate por customer_id crescente
SELECT
    customer_id,
    faturamento_total,
    frequencia,
    ticket_medio,
    diversidade_categorias
FROM metricas_cliente
WHERE diversidade_categorias >= 13
ORDER BY ticket_medio DESC, customer_id ASC
LIMIT 10;"""
conn = psycopg2.connect(conn_string)
cur = conn.cursor()
cur.execute(query)
resultado = cur.fetchall()

for linha in resultado:
    print(linha)

conn.close()

(22, Decimal('1087838.44'), 26, Decimal('41839.94'), 14)
(1477, Decimal('916262.58'), 22, Decimal('41648.30'), 14)
(929, Decimal('1082775.89'), 26, Decimal('41645.23'), 14)
(1116, Decimal('655737.20'), 16, Decimal('40983.58'), 14)
(1691, Decimal('815471.30'), 20, Decimal('40773.57'), 14)
(774, Decimal('726127.99'), 18, Decimal('40340.44'), 14)
(1470, Decimal('1040553.09'), 26, Decimal('40021.27'), 14)
(1599, Decimal('997616.46'), 25, Decimal('39904.66'), 14)
(965, Decimal('677297.78'), 17, Decimal('39841.05'), 14)
(1722, Decimal('1146455.22'), 29, Decimal('39532.94'), 14)


- O cliente 22 lidera com ticket médio de R$ 41.839,94,

  distribuído em 26 pedidos (faturamento total de R$ 1,09 milhão)

- Todos os 10 têm diversidade 14. O teto máximo do catálogo, confirmando o que já tínhamos descoberto: o filtro de elite não está realmente "filtrando" pela diversidade (quase todo mundo bate isso), quem decide o ranking é mesmo o ticket médio

- A faixa de ticket médio do grupo vai de R$ 39.532,94 (último, cliente 1722)

  a R$ 41.839,94 (primeiro), uma diferença de só 5,5% entre o topo e a base do grupo, ou seja, é um grupo bem coeso, não tem um "super outlier" destoando muito dos outros 9

**categoria mais comprada entre os 10 clientes de elite**

pegar a lista dos 10 clientes que já encontramos, e agora somar quantity de order_items, mas filtrando só os pedidos desses 10 clientes, agrupando por categoria.

In [31]:
query_categoria = """
WITH pedidos_cliente AS (
    SELECT customer_id, SUM(total) AS faturamento_total, COUNT(*) AS frequencia
    FROM orders
    GROUP BY customer_id
),
categorias_cliente AS (
    SELECT o.customer_id, COUNT(DISTINCT p.category_id) AS diversidade_categorias
    FROM orders o
    JOIN order_items oi      ON oi.order_id = o.id
    JOIN product_variants pv ON pv.id = oi.product_variant_id
    JOIN products p          ON p.id = pv.product_id
    GROUP BY o.customer_id
),
metricas_cliente AS (
    SELECT pc.customer_id, ROUND(pc.faturamento_total / pc.frequencia, 2) AS ticket_medio,
           cc.diversidade_categorias
    FROM pedidos_cliente pc
    JOIN categorias_cliente cc ON cc.customer_id = pc.customer_id
),
clientes_elite AS (
    SELECT customer_id
    FROM metricas_cliente
    WHERE diversidade_categorias >= 13
    ORDER BY ticket_medio DESC, customer_id ASC
    LIMIT 10
)
SELECT
    p.category_id,
    c.name AS nome_categoria,
    SUM(oi.quantity) AS qtd_total_itens
FROM clientes_elite ce
JOIN orders o             ON o.customer_id = ce.customer_id
JOIN order_items oi       ON oi.order_id = o.id
JOIN product_variants pv  ON pv.id = oi.product_variant_id
JOIN products p           ON p.id = pv.product_id
JOIN categories c         ON c.id = p.category_id
GROUP BY p.category_id, c.name
ORDER BY qtd_total_itens DESC
LIMIT 1;
"""

conn = psycopg2.connect(conn_string)
cur = conn.cursor()
cur.execute(query_categoria)
resultado = cur.fetchall()
for linha in resultado:
    print(linha)
conn.close()

(8, 'Hélices', 492)


## **Questão 4.2 - Explicação:**


**1. Como cheguei às categorias mais vendidas (mapeamento da cadeia de chaves):**
Para descobrir a categoria, percebi que essa informação não estava fácil ali na tabela de clientes nem no pedido. Precisei cruzando 5 tabelas até chegar lá:
Comecei em orders onde tenho o customer_id, cruzei com order_items pelo ID do pedido que é onde está a quantidade vendida de cada item. Daí, liguei os itens à product_variants pelo ID da variante e, depois, de variantes para a products (onde finalmente aparece o category_id). No fim, só fiz um JOIN com categories para pegar o nome bonitinho da categoria em vez de mostrar só o número do ID.
Com essa sequência de JOINs, consegui ver cada item vendido com a sua respectiva categoria, o que me permitiu agrupar tudo e fazer o SUM(quantity) sem erro

**2. Lógica utilizada para filtrar os clientes com diversidade mínima:**
Para calcular a diversidade, usei COUNT(DISTINCT products.category_id) agrupando por cliente. Usei o DISTINCT pois se eu não usasse ele, a contagem ia inflar, porque contaria a mesma categoria várias vezes caso o cliente comprasse dois ou três produtos do mesmo tipo. Com o DISTINCT, a categoria só conta uma única vez por cliente, não importa quantos itens daquela categoria ele levou.
Depois, criei uma CTE que chamei de clientes_elite e apliquei a regra de ter comprado em pelo menos 13 categorias distintas,, isolando certinho só quem batia esse critério antes de ir para a etapa de ranking.

**3. Como garanti que a contagem de itens refletisse apenas o grupo Top 10:**
A CTE clientes_elite já contém sozinha o filtro de diversidade e o ranking dos 10 maiores Ticket Médio, ou seja, ela é, por definição, a lista final dos 10 clientes-alvo. Na consulta final, uso um JOIN clientes_elite ce ON o.customer_id = ce.customer_id antes de somar quantity. Esse JOIN funciona como um filtro: só entram no cálculo os pedidos cujo customer_id está presente nessa lista de 10. Sem isso, a soma de quantity incluiria itens comprados por qualquer um dos 2.000 clientes da base, não só os 10 de elite.

## **Questão 5 - Dimensão de Calendário**

Construa uma dimensão de datas (Calendário) em SQL.

Realize um LEFT JOIN do calendário com a tabela de vendas para análise agregada

In [32]:
import os
import psycopg2

DIRETORIO_CSVS = "/content/drive/MyDrive/lh-nautical/1-lh_nautical_csv"
CONN_STRING = userdata.get('NEON_CONNECTION_STRING')

## **Questão 5.1 - Dimensão de Calendário + Vendas por Dia da Semana (lojas físicas)**

- Gera uma linha para CADA dia entre a menor e a maior data de orders,inclusive dias em que não houve nenhuma venda registrada.

- Soma de vendas por dia, considerando SÓ lojas físicas (channel = 'pos')

- LEFT JOIN: mantém TODOS os dias do calendário, mesmo os que não têm correspondência em vendas_diarias_pos (dias sem venda).

- COALESCE troca o NULL resultante (dia sem venda) por 0.

 - Traduz o número do dia da semana (0=Domingo ... 6=Sábado, padrão do Postgres) para o nome em português.

In [33]:
query = """
WITH calendario AS (
    SELECT generate_series(
        (SELECT MIN(created_at)::date FROM orders),
        (SELECT MAX(created_at)::date FROM orders),
        interval '1 day'
    )::date AS data
),

vendas_diarias_pos AS (
    SELECT
        created_at::date AS data,
        SUM(total) AS valor_venda
    FROM orders
    WHERE channel = 'pos'
    GROUP BY created_at::date
),

calendario_vendas AS (
    SELECT
        cal.data,
        COALESCE(vd.valor_venda, 0) AS valor_venda,
        EXTRACT(DOW FROM cal.data) AS dia_semana_num
    FROM calendario cal
    LEFT JOIN vendas_diarias_pos vd ON vd.data = cal.data
),

calendario_vendas_nome AS (
    SELECT
        data,
        valor_venda,
        dia_semana_num,
        CASE dia_semana_num
            WHEN 0 THEN 'Domingo'
            WHEN 1 THEN 'Segunda-feira'
            WHEN 2 THEN 'Terça-feira'
            WHEN 3 THEN 'Quarta-feira'
            WHEN 4 THEN 'Quinta-feira'
            WHEN 5 THEN 'Sexta-feira'
            WHEN 6 THEN 'Sábado'
        END AS nome_dia_semana
    FROM calendario_vendas
)

SELECT
    nome_dia_semana,
    ROUND(AVG(valor_venda), 2) AS media_vendas,
    COUNT(*) AS qtd_dias_no_periodo
FROM calendario_vendas_nome
GROUP BY nome_dia_semana, dia_semana_num
ORDER BY dia_semana_num;

"""

conn = psycopg2.connect(conn_string)
cur = conn.cursor()
cur.execute(query)
resultado = cur.fetchall()
for linha in resultado:
    print(linha)
conn.close()

('Domingo', Decimal('157616.13'), 365)
('Segunda-feira', Decimal('158241.15'), 365)
('Terça-feira', Decimal('166118.83'), 365)
('Quarta-feira', Decimal('173605.44'), 366)
('Quinta-feira', Decimal('157154.32'), 366)
('Sexta-feira', Decimal('170193.68'), 365)
('Sábado', Decimal('164858.27'), 365)


- Pior dia: Quinta-feira, com média de R$ 157.154,32, não é o Domingo que o estagiário fictício da história apontou

- O Domingo, na verdade, fica em segundo pior (R$ 157.616,13)

 bem próximo da Quinta, a diferença entre os dois é de menos de R$ 500, praticamente empatados

- O melhor dia é Quarta-feira (R$ 173.605,44),

 seguido de Sexta-feira (R$ 170.193,68)

- A diferença de 365 vs 366 dias em algumas linhas é só o efeito de anos bissextos dentro do período 2020-2026 (2020 e 2024 são bissextos),  não é erro, é matemática do calendário mesmo

## **Questão 5.2 — Explicação sobre a Dimensão de Calendário**

**1. Por que precisamos usar uma tabela de datas (calendário) em vez de agrupar direto a tabela de vendas?**
A tabela orders só registra quando uma venda efetivamente acontece. Se a loja abriu em um domingo e não vendeu absolutamente nada, não existe linha gerada no banco para aquele dia . ele simplesmente não existe na tabela de pedidos.
Se eu fizer um GROUP BY direto na orders, o banco só vai enxergar os dias em que houve faturamento. Para corrigir isso, criamos uma dimensão de calendário com generate_serie. . Esse calendário funciona como uma régua com todas as datas do período. Ao fazer um LEFT JOIN do calendário com a tabela de vendas, eu trago todos os dias reais e forço os dias sem faturamento a aparecerem preenchidos com zero, refletindo o que realmente aconteceu na loja física.

**2. O que aconteceria com a média de vendas se um dia da semana tivesse muitos dias sem nenhuma venda?**
Sem a tabela de calendário, a média daquele dia da semana seria calculada dividindo o total faturado apenas pela quantidade de dias em que houve venda. O denominador da média ficaria artificialmente menor do que deveria e os dias de venda zero seriam ignorados.
Isso infla a média para cima e gera uma falsa impressão de bom desempenho. Esse foi exatamente o equívoco do estagiário no enunciado: ele disse que o Domingo tinha uma média ótima de R$ 5.000,00

 porque só contou os domingos em que a loja vendeu. Ele esqueceu de incluir no cálculo os domingos em que a loja esteve aberta, teve custo de operação e faturou R$ 0,00.

## **Questão 6 — Previsão de Demanda**
### Bússola de Bordo 702 — Baseline de Média Móvel

**Premissas do enunciado:**
- Treino: até 31/12/2025
- Teste: 1º trimestre de 2026 (Jan, Fev, Mar)
- Previsão em base mensal
- Baseline: média móvel dos últimos 3 meses (só dados anteriores à data prevista)
- Métrica: MAE

Vamos cruzar `order_items` (o que foi vendido) com `orders` (quando foi vendido) e `product_variants` (qual produto cada variante representa)

In [34]:
import pandas as pd

orders = pd.read_csv('/content/drive/MyDrive/lh-nautical/1-lh_nautical_csv/orders.csv')
order_items = pd.read_csv('/content/drive/MyDrive/lh-nautical/1-lh_nautical_csv/order_items.csv')
product_variants = pd.read_csv('/content/drive/MyDrive/lh-nautical/1-lh_nautical_csv/product_variants.csv')
products = pd.read_csv('/content/drive/MyDrive/lh-nautical/1-lh_nautical_csv/products.csv')

Antes de filtrar, é importante checar se o nome do produto é realmente único na base — não é sempre seguro assumir isso sem checar.

In [35]:
match = products[products['name'] == 'Bússola de Bordo 702']
match

,id,name,description,brand_id,category_id,ncm_code,unit_of_measure,is_active,created_at,updated_at
73,74,Bússola de Bordo 702,Bússola magnética líquida com iluminação,12,8,18607512,UN,True,2025-01-27 17:41:23,2025-04-09 17:31:25
239,240,Bússola de Bordo 702,Bússola magnética líquida com iluminação,8,7,91287147,UN,True,2026-06-22 00:02:10,2026-11-18 21:03:07


**Achado:** existem 2 produtos com o nome exato "Bússola de Bordo 702" — `id=74` (categoria Hélices) e `id=240` (categoria Âncoras). Nenhuma das duas categorias é coerente com o que o produto realmente é (um instrumento de navegação), o que sugere que a categorização nesse dataset sintético foi atribuída de forma aleatória, não intencional — não dá para confirmar isso como certeza absoluta, só como hipótese plausível.
Além disso, os dois id tem vendas nas datas relacionadas que utilizaremos para previsão, ou seja descartar um ou outro por verificação de data não sera possivel.
São produtos genuinamente diferentes (marcas e categorias diferentes), então a decisão correta é tratá-los como itens distintos, nunca somar as vendas dos dois.


### **Volume de vendas de cada candidato (critério de decisão)**

In [36]:
for pid in match['id']:
    variantes_produto = product_variants[product_variants['product_id'] == pid]['id'].tolist()
    qtd_vendida = order_items[order_items['product_variant_id'].isin(variantes_produto)]['quantity'].sum()
    qtd_itens_pedido = len(order_items[order_items['product_variant_id'].isin(variantes_produto)])
    print(f"Produto id={pid}: {len(variantes_produto)} variante(s), "
          f"{qtd_itens_pedido} linhas de pedido, {qtd_vendida} unidades vendidas no total")

Produto id=74: 2 variante(s), 330 linhas de pedido, 1760 unidades vendidas no total
Produto id=240: 1 variante(s), 142 linhas de pedido, 783 unidades vendidas no total


Vamos confirmarque se os dois têm vendas em TODA a janela de treino/teste

Antes de decidir, é importante confirmar que a escolha não foi "forçada" por falta de dado em um dos dois candidatos dentro do período que a questão pede.

In [37]:
print('Vendas mensais dos dois produtos candidatos, dentro da janela de treino (2025) e teste (Jan-Mar/2026):\n')

for pid in match['id']:
    variantes_produto = product_variants[product_variants['product_id'] == pid]['id'].tolist()
    vendas_produto = order_items[order_items['product_variant_id'].isin(variantes_produto)]
    vendas_produto = vendas_produto.merge(orders[['id', 'created_at']], left_on='order_id', right_on='id')
    vendas_produto['created_at'] = pd.to_datetime(vendas_produto['created_at'])
    vendas_produto['ano_mes'] = vendas_produto['created_at'].dt.to_period('M')

    serie_produto = vendas_produto.groupby('ano_mes')['quantity'].sum()

    print(f'--- Produto id={pid} ---')
    print('2025 (treino):')
    print(serie_produto['2025-01':'2025-12'].to_string())
    print('2026 Q1 (teste):')
    print(serie_produto['2026-01':'2026-03'].to_string())
    print()

Vendas mensais dos dois produtos candidatos, dentro da janela de treino (2025) e teste (Jan-Mar/2026):

--- Produto id=74 ---
2025 (treino):
ano_mes
2025-01    25
2025-02    21
2025-03    70
2025-04    23
2025-05    10
2025-06     7
2025-07    14
2025-08     5
2025-09    21
2025-10    26
2025-11    36
2025-12    14
Freq: M
2026 Q1 (teste):
ano_mes
2026-01    69
2026-02    42
2026-03    45
Freq: M

--- Produto id=240 ---
2025 (treino):
ano_mes
2025-01    32
2025-02    11
2025-03     7
2025-04    15
2025-05    14
2025-06    10
2025-07     5
2025-08    18
2025-09    10
2025-10     8
2025-11    24
2025-12     8
Freq: M
2026 Q1 (teste):
ano_mes
2026-01    10
2026-02    26
2026-03    15
Freq: M



**Confirmado:** os dois produtos têm vendas em todos os meses da janela pedida, nenhum some por falta de dado. Isso significa que os dois eram candidatos igualmente válidos, e por isso decidimos ir além: em vez de escolher só um e descartar o outro, vamos rodar o baseline para os dois e comparar os resultados. Isso é mais rigoroso do que decidir só por um critério indireto, volume histórico, sem testar de fato.

**Passo 3 — Função reutilizável: construir série mensal + baseline**

Como vamos repetir o mesmo processo para os dois produtos, vamos escrever funções reutilizáveis em vez de copiar e colar o código duas vezes.

In [38]:
def construir_serie_mensal(product_id):
    variantes = product_variants[product_variants['product_id'] == product_id]['id'].tolist()
    vendas = order_items[order_items['product_variant_id'].isin(variantes)]
    vendas = vendas.merge(orders[['id', 'created_at']], left_on='order_id', right_on='id')
    vendas['created_at'] = pd.to_datetime(vendas['created_at'])
    vendas['ano_mes'] = vendas['created_at'].dt.to_period('M')

    serie = vendas.groupby('ano_mes')['quantity'].sum()
    todos_os_meses = pd.period_range(serie.index.min(), '2026-03', freq='M')
    return serie.reindex(todos_os_meses, fill_value=0)


def media_movel_3_meses(serie, mes_alvo):
    posicao = serie.index.get_loc(mes_alvo)
    return serie.iloc[posicao - 3 : posicao].mean()

Aqui na primeira função construimos uma série de vendas mensais (unidades) de um produto, garantindo que todos os meses existam (mesmo os com 0 vendas), do primeiro mês com venda até 2026-03.
  
Na segunda : fizemos a média dos 3 meses imediatamente anteriores a mes_alvo, nunca olha para o próprio mês ou para o futuro (evita data leakage).
  

**Rodar o baseline para os DOIS produtos e comparar**

In [39]:
meses_teste = ['2026-01', '2026-02', '2026-03']
resultados = {}

for pid in [74, 240]:
    serie = construir_serie_mensal(pid)
    previsoes = {m: media_movel_3_meses(serie, m) for m in meses_teste}
    reais = {m: serie[m] for m in meses_teste}
    erros_absolutos = [abs(reais[m] - previsoes[m]) for m in meses_teste]
    mae = sum(erros_absolutos) / len(erros_absolutos)
    media_real = sum(reais.values()) / len(reais)

    resultados[pid] = {
        'previsoes': previsoes,
        'reais': reais,
        'mae': mae,
        'soma_previsao': sum(previsoes.values()),
        'media_real': media_real,
        'mae_percentual': 100 * mae / media_real,
    }

    print(f'=== Produto id={pid} ===')
    for m in meses_teste:
        print(f'  {m}: previsto={previsoes[m]:.2f}  real={reais[m]}  erro_abs={abs(reais[m]-previsoes[m]):.2f}')
    print(f'  Soma da previsão (arredondada): {round(sum(previsoes.values()))}')
    print(f'  MAE: {mae:.2f}  ({resultados[pid]["mae_percentual"]:.1f}% da média real)')
    print()

=== Produto id=74 ===
  2026-01: previsto=25.33  real=69  erro_abs=43.67
  2026-02: previsto=39.67  real=42  erro_abs=2.33
  2026-03: previsto=41.67  real=45  erro_abs=3.33
  Soma da previsão (arredondada): 107
  MAE: 16.44  (31.6% da média real)

=== Produto id=240 ===
  2026-01: previsto=13.33  real=10  erro_abs=3.33
  2026-02: previsto=14.00  real=26  erro_abs=12.00
  2026-03: previsto=14.67  real=15  erro_abs=0.33
  Soma da previsão (arredondada): 42
  MAE: 5.22  (30.7% da média real)



**Análise comparativa entre os dois produtos**

In [40]:
print('Comparativo final:\n')
print(f'{"Produto":10s} {"MAE (unid.)":>12s} {"MAE (% da média)":>18s} {"Soma prevista":>15s}')
for pid in [74, 240]:
    r = resultados[pid]
    print(f'{"id="+str(pid):10s} {r["mae"]:>12.2f} {r["mae_percentual"]:>17.1f}% {round(r["soma_previsao"]):>15d}')

Comparativo final:

Produto     MAE (unid.)   MAE (% da média)   Soma prevista
id=74             16.44              31.6%             107
id=240             5.22              30.7%              42


Em valor absoluto, o MAE do `id=74` (16,44) parece muito pior que o do `id=240` (5,22). Mas isso é enganoso, porque os dois produtos têm escalas de venda diferentes (`id=74` vende ~52 unid./mês em média no teste, `id=240` vende ~17 unid./mês). Quando normalizamos o erro pela média real (MAE como % da média), os dois ficam praticamente empatados: **31,6% vs 30,7%**. Ou seja, o baseline erra proporcionalmente quase a mesma coisa nos dois produtos — a limitação identificada é do **método** (média móvel simples), não uma peculiaridade de um produto específico ser mais difícil de prever que o outro.

**Decisão final para responder oficialmente a Questão 6:** usar o `id=74`, por ter o maior volume histórico de vendas (330 unidades vs 142), o que dá uma base mais robusta de dados — mas documentamos aqui que o comportamento do baseline é consistente entre os dois candidatos, reforçando que a conclusão sobre a adequação do método (Passo 6) vale para o produto de forma geral, não é um resultado isolado de uma escolha arbitrária.

## **Questão 6.2 (validação oficial): soma da previsão para `id=74`**

In [41]:
soma_previsao_oficial = resultados[74]['soma_previsao']
print(f'Soma das previsões (bruta): {soma_previsao_oficial:.2f}')
print(f'Soma arredondada para inteiro: {round(soma_previsao_oficial)}')

Soma das previsões (bruta): 106.67
Soma arredondada para inteiro: 107


## **Questão 6.3: o baseline é adequado**?

**1. Como o baseline foi construído?**
O baseline usa média móvel simples de 3 meses: para prever um mês qualquer, calculamos a média das quantidades vendidas nos 3 meses imediatamente anteriores a ele. a previsão de Fevereiro/2026 já usa o valor real de Janeiro/2026 que se tornou passado no momento de prever Fevereiro, e a previsão de Março/2026 usa os valores reais de Janeiro e Fevereiro/2026.

**2. Como evitou data leakage?** A cada previsão, a janela dos 3 meses usados como base é sempre estritamente anterior ao mês sendo previsto, nunca inclui o próprio mês-alvo nem meses posteriores a ele. A função media_movel_3_meses usa a posição do mês na série e recorta exatamente os 3 meses antes dessa posição, garantindo isso de forma automática e sem risco de erro manual.

**3. Uma limitação do modelo proposto ?** A principal limitação é que a média móvel reage com atraso a mudanças bruscas de patamar , ela só aprende que as vendas subiram depois que a subida já aconteceu. Isso ficou evidente no teste com o id=74: o erro de Janeiro/2026 ,43,67 unidades, foi mais de 10x maior que os erros de Fevereiro e Março 2,33 e 3,33, porque Janeiro teve um salto de vendas que a média dos meses anteriores não conseguiu antecipar. Ao rodar o mesmo baseline também no produto homônimo id=240 como checagem cruzada, e o padrão se confirmou: o erro relativo  ficou praticamente igual nos dois produtos 31,6% vs 30,7%, mostrando que essa é uma limitação do método, não uma particularidade de um produto específico. Diante disso, o baseline é parcialmente adequado, funciona como um primeiro patamar de comparação, simples e barato de calcular, mas não é confiável para produtos com demanda sazonal ou picos abruptos como parece ser o caso da Bússola de Bordo 702 , nesses casos, um erro médio de 32% em relação à venda real é alto demais para guiar decisões de compra com segurança, sob risco de repetir o mesmo problema do Colete Salva-Vidas citado na história, estoque zerando por falta de antecipação da demanda.

## **Questão 7 — Sistema de Recomendação**

**Objetivo:** encontrar os 5 produtos mais similares ao "Motor de Popa 1949", com base na similaridade de comportamento de compra dos clientes (não na descrição/categoria do produto).

**Bibliotecas permitidas:** pandas, numpy, sklearn (opcional, para cosine similarity).

**Carregar os dados**

In [42]:
import pandas as pd
import numpy as np

orders = pd.read_csv('/content/drive/MyDrive/lh-nautical/1-lh_nautical_csv/orders.csv')
order_items = pd.read_csv('/content/drive/MyDrive/lh-nautical/1-lh_nautical_csv/order_items.csv')
product_variants = pd.read_csv('/content/drive/MyDrive/lh-nautical/1-lh_nautical_csv/product_variants.csv')
products = pd.read_csv('/content/drive/MyDrive/lh-nautical/1-lh_nautical_csv/products.csv')

Vimos na Questão 6 que nomes de produto podem se repetir nessa base, então vale sempre checar antes de assumir.

In [43]:
match = products[products['name'] == 'Motor de Popa 1949']
match

,id,name,description,brand_id,category_id,ncm_code,unit_of_measure,is_active,created_at,updated_at
179,180,Motor de Popa 1949,Motor de popa 4 tempos para embarcações de lazer,7,13,22146679,UN,True,2024-11-01 07:14:54,2024-11-10 04:23:05


Confirmado: único produto com esse nome, `id=180`. Diferente da Questão 6, aqui não há ambiguidade — seguimos direto.

 **Montar a matriz de interação Usuário × Produto**

Precisamos de uma tabela em que cada linha é um cliente, cada coluna é um produto, e o valor da célula é `1` se aquele cliente já comprou aquele produto (não importa quantas vezes ou em qual quantidade) e `0` caso contrário.

 a informação de "o que o cliente comprou" não está pronta em nenhuma tabela isolada, precisamos cruzar `order_items` (o que foi comprado, por variante) com `orders` (de qual cliente) e com `product_variants` (a variante pertence a qual produto).

In [44]:
# Cadeia de JOINs: order_items para- orders (traz customer_id) para- product_variants (traz product_id)
oi = order_items.merge(orders[['id', 'customer_id']], left_on='order_id', right_on='id')
oi = oi.merge(product_variants[['id', 'product_id']], left_on='product_variant_id', right_on='id', suffixes=('', '_pv'))

# Pares únicos,ignora quantidade e repetições,
# só interessa se o cliente comprou aquele produto alguma vez.
interacoes = oi[['customer_id', 'product_id']].drop_duplicates()

print(f'Total de pares (cliente, produto) únicos: {len(interacoes)}')
print(f'Clientes distintos: {interacoes["customer_id"].nunique()}')
print(f'Produtos distintos com ao menos 1 venda: {interacoes["product_id"].nunique()}')

Total de pares (cliente, produto) únicos: 135508
Clientes distintos: 2000
Produtos distintos com ao menos 1 venda: 500


In [45]:
# pivot_table transforma o formato de uma linha por par cliente-produto
# fill_value=0 preenche com 0 as combinações cliente-produto que não existem
# ou seja, o cliente nunca comprou aquele produto.

matriz_usuario_produto = interacoes.assign(valor=1).pivot_table(
    index='customer_id', columns='product_id', values='valor', fill_value=0
)

print('Shape da matriz (clientes x produtos):', matriz_usuario_produto.shape)
matriz_usuario_produto.head()

Shape da matriz (clientes x produtos): (2000, 500)


product_id,1,2,3,4,5,6,7,8,9,10,...,491,492,493,494,495,496,497,498,499,500
customer_id,,,,,,,,,,,,,,,,,,,,,
1,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
2,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,...,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
3,0.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
4,0.00,0.00,0.00,0.00,1.00,0.00,1.00,1.00,0.00,1.00,...,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00
5,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,...,0.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00



**Calcule a Similaridade de Cosseno (Cosine Similarity) entre os vetores de cada produto.**

In [46]:
from sklearn.metrics.pairwise import cosine_similarity

# .T : produtos passam a ser as linhas, o que é o que
# cosine_similarity: compara linha com linha.
matriz_similaridade = cosine_similarity(matriz_usuario_produto.T)

# Reconstrói como DataFrame, com os product_id como índice E como coluna,
# pra facilitar consultar a similaridade do produto X com o produto Y.
sim_df = pd.DataFrame(
    matriz_similaridade,
    index=matriz_usuario_produto.columns,
    columns=matriz_usuario_produto.columns
)

print('Shape da matriz de similaridade (produtos x produtos):', sim_df.shape)

Shape da matriz de similaridade (produtos x produtos): (500, 500)


In [47]:
# Motor de Popa 1949
PRODUTO_ALVO = 180

similares = sim_df[PRODUTO_ALVO].drop(PRODUTO_ALVO).sort_values(ascending=False).head(5)

print('Top 5 produtos mais similares ao Motor de Popa 1949:\n')
for product_id, score in similares.items():
    nome = products[products['id'] == product_id]['name'].values[0]
    print(f'  id={product_id:4d}   similaridade={score:.4f}   {nome}')

Top 5 produtos mais similares ao Motor de Popa 1949:

  id= 389   similaridade=0.2566   Motor de Popa 5331
  id= 295   similaridade=0.2562   Cabo Náutico 2105
  id=  75   similaridade=0.2558   Vela Mestra 1913
  id= 337   similaridade=0.2393   Cabo Náutico 9048
  id=  55   similaridade=0.2377   GPS Plotter 6249


## **Questão 7.2 - Validação**

**Qual é o nome do produto com maior similaridade em relação ao "Motor de Popa 1949"? :** Motor de Popa 5331 (similaridade ≈ 0,2566)

O produto mais similar é justamente **outro Motor de Popa**, isso é um bom sinal de que a lógica capturou um padrão real de comportamento de compra clientes que compram um motor de popa tendem a considerar outro também, e não apenas ruído.

## **Questão 7.3 — Apoio para a explicação**

**1. Como a matriz de interação foi construíd**a?

Para montar a matriz, primeiro cruzei a order_items com a orders para mapear qual cliente fez cada pedido e com a product_variants para chegar ao produto "pai" de cada variante.

Em seguida, apliquei o .drop_duplicates() nos pares customer_id, product_id. Esse passo foi fundamental, pois ele já garante a regra do enunciado de ignorar a quantidade comprada, mantendo apenas a presença ou ausência do produto.
Por fim, usei o pivot_table para transformar essa lista em uma matriz, cada linha virou um cliente, cada coluna um produto, preenchendo com 1 onde o cliente comprou e 0 fill_value=0 onde não comprou.

**2. O que significa a Similaridade de Cosseno nesse context**o?

Nesta análise, cada produto é representado por um vetor com a posição de todos os clientes da base com 1 para quem comprou e 0 para quem não comprou. A Similaridade de Cosseno mede a proximidade do ângulo entre dois desses vetores. Quanto mais próximo de 1, mais parecidos são os públicos daqueles dois produtos.

Vale destacar que não estamos comparando características físicas dos produtos como preço, marca ou categoria, mas sim o comportamento dos clientes. Dois produtos são considerados similares quando o mesmo grupo de pessoas costuma comprar ambos.

**3. Qual é uma limitação desse método de recomendação?**

Esse método tem duas limitações bem claras:
Problema da Partida Fria Cold Start, Produtos novos que acabaram de entrar no catálogo e ainda não possuem histórico de vendas ficam com o vetor Zerado. Por causa disso, o algoritmo nunca vai recomendá-los, mesmo que façam todo sentido do ponto de vista do negócio.

Baixa sobreposição da base: Os índices de similaridade ficaram abaixo de 0,26. Isso acontece porque a base tem cerca de 500 produtos e a maioria dos clientes compra poucos itens. Como a sobreposição de compradores entre produtos diferentes é pequena, essas recomendações devem ser vistas como um sinal complementar de suporte, e não como uma certeza absoluta.